# WTI Producer Hedge Simulator

I built this notebook to look at futures from the producer side: how much can WTI futures reduce the risk of falling crude prices?

The example producer expects to sell 100,000 barrels per month. I start with the basic hedge, then look at Midland/Cushing basis risk and a data-driven hedge ratio.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import yfinance as yf
from pandas_datareader import data as web

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

from src.hedge_engine import HedgeAssumptions, prepare_monthly_market_data, compare_hedge_ratios, stress_test
from src.basis_risk import compare_basis_scenarios, simulate_basis_scenario, BasisHedgeAssumptions
from src.min_variance import minimum_variance_hedge_ratio, hedge_ratio_diagnostics, rolling_minimum_variance_hedge_ratio, contracts_for_hedge_ratio


## Load the WTI prices

The physical side uses WTI Cushing spot prices. The hedge side uses WTI futures prices. One CL futures contract represents 1,000 barrels.

In [ ]:
spot = web.DataReader('DCOILWTICO', 'fred', '2018-01-01')['DCOILWTICO']
raw = yf.download('CL=F', start='2018-01-01', auto_adjust=False, progress=False)
futures = raw['Close'].iloc[:, 0] if isinstance(raw.columns, pd.MultiIndex) else raw['Close']
market = prepare_monthly_market_data(spot, futures)
market.tail()


## Compare different hedge sizes

The hedge ratio is the percentage of expected production covered by futures. I compare 0%, 25%, 50%, 75%, and 100%.

In [ ]:
assumptions = HedgeAssumptions(monthly_production_bbl=100_000)
summary, simulations = compare_hedge_ratios(market, assumptions=assumptions)
summary


## Stress test: what if oil falls 25%?

This checks how much a 75% hedge changes the result during a large crude-price drop.

In [ ]:
stress_test(spot_price=75, futures_entry=76, spot_shock_pct=-0.25, hedge_ratio=0.75, assumptions=assumptions)


## Midland vs. Cushing basis risk

Cushing is the delivery point tied to NYMEX WTI futures. Midland is a major Permian Basin crude-pricing location.

The two prices usually move together, but not perfectly. The difference between them is basis, and a change in that difference creates basis risk.

In [ ]:
basis_scenarios = compare_basis_scenarios(
    realized_basis_values=(1, 0, -1, -3, -5, -10),
    basis_hedge_ratios=(0, 0.50, 1.00),
    cushing_futures_entry=75,
    cushing_futures_exit=60,
    cushing_spot_exit=60,
    locked_basis_per_bbl=-1,
    flat_price_hedge_ratio=1.0,
    assumptions=assumptions,
)
basis_scenarios[[
    'basis_hedge_ratio',
    'realized_midland_basis',
    'basis_swap_pnl',
    'residual_revenue_risk',
]]


### What this means

A WTI futures hedge can reduce the overall oil-price move and still leave location risk. If Midland weakens relative to Cushing, the producer can still receive less than expected for the physical crude.

## Let the historical data choose a hedge size

Here I use the historical relationship between spot and futures price changes to estimate the hedge ratio that minimized residual price movement.

In [ ]:
mv_ratio = minimum_variance_hedge_ratio(market)
mv_contracts = contracts_for_hedge_ratio(100_000, mv_ratio)
mv_ratio, mv_contracts


In [ ]:
hedge_ratio_diagnostics(market, mv_contracts / 100)


In [ ]:
rolling_mv = rolling_minimum_variance_hedge_ratio(market, window=24)
rolling_mv.tail()
